### Imports + paths + load data

In [18]:
import sys
print(sys.executable)

import warnings
warnings.filterwarnings("ignore", message="`artifact_path` is deprecated")
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")

from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter("ignore", InsecureRequestWarning)



c:\Users\kimwo\software engineering\year 4\LP3\4DT907 - Project\machine_learning_project_4dt907\.venv\Scripts\python.exe


In [19]:
import numpy as np
import pandas as pd
from pathlib import Path

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Notebook location: ml/models/research/A5/Bootstrap/
SCORES_PATH = Path("../../../../data/processed/20190108_scores_and_weak_links.csv")
FEATURES_PATH = Path("../../../../data/processed/AimoScore_WeakLink_big_scores.csv")
REG_PATH = Path("../../../../data/raw/AimoScore_WeakLink_big_scores.xls")

scores = pd.read_csv(SCORES_PATH)
big = pd.read_csv(FEATURES_PATH)
df_reg = pd.read_excel(REG_PATH)

print("scores:", scores.shape)
print("big:", big.shape)
print("regression:", df_reg.shape)

display(scores.head(2))
display(big.head(2))
display(df_reg.head(2))


scores: (2096, 17)
big: (2094, 43)
regression: (2094, 43)


,ID,Date,SCORE,ForwardHead,LeftArmFallForward,RightArmFallForward,LeftShoulderElevation,RightShoulderElevation,ExcessiveForwardLean,LeftAsymmetricalWeightShift,RightAsymmetricalWeightShift,LeftKneeMovesInward,RightKneeMovesInward,LeftKneeMovesOutward,RightKneeMovesOutward,LeftHeelRises,RightHeelRises
0,cf0e95ab-d6ec-475a-9cc0-c9444b8dd080.Kinect,2018-07-19,0.612532,0.31,1.34,1.15,0.33,0.49,0.00,0.0,0.08,0.0,0.0,0.0,0.77,0.0,0.0
1,6210cc6d-575d-4e9f-bab1-4fe1da2a30a3.Kinect,2018-07-30,0.388809,1.49,1.42,1.39,0.00,0.00,1.08,0.1,0.00,0.0,0.0,0.0,0.40,0.0,0.0


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


,AimoScore,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,...,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation,EstimatedScore,ID
0,0.323667,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,...,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747,0.209947,0003cdcc-86ed-494a-a3b5-90d09e96e06b.Kinect
1,0.323699,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,...,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207,0.457198,003115c4-bdb8-491c-b571-8fcebdecf8ed.Kinect


## Common utilities

In [20]:
from dataclasses import dataclass
from typing import Tuple, Dict, List

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import joblib


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def regression_metrics(y_true, y_pred) -> Dict[str, float]:
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": rmse(y_true, y_pred),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def make_bootstrap_indices(n: int, seed: int) -> np.ndarray:
    """
    Bootstrap sample indices (with replacement), size n.
    """
    rng = np.random.default_rng(seed)
    return rng.integers(low=0, high=n, size=n, endpoint=False)


def oob_mask(n: int, boot_idx: np.ndarray) -> np.ndarray:
    """
    Out-of-bag mask: True for samples NOT included in bootstrap draw.
    """
    mask = np.ones(n, dtype=bool)
    mask[np.unique(boot_idx)] = False
    return mask


def safe_mkdir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


## MLFlow setup

In [1]:
import mlflow
import mlflow.sklearn
from dotenv import load_dotenv
from os import getenv

load_dotenv("../../../../../../../env")  # if your .env is elsewhere: load_dotenv("path/to/.env")

tracking_uri = getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)
else:
    print("WARNING: MLFLOW_TRACKING_URI not set. MLflow will use its default local tracking location.")

EXPERIMENT_NAME = "A5_bootstrap_regression"
mlflow.set_experiment(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("MLflow experiment:", EXPERIMENT_NAME)


KeyboardInterrupt: 

## Regression dataset prep (same approach as A4)

In [22]:
# Target
y_reg = df_reg["AimoScore"].astype(float)

# Features
X_reg = df_reg.drop(columns=["AimoScore", "EstimatedScore", "ID"], errors="ignore")
X_reg = X_reg.apply(pd.to_numeric, errors="coerce")  # keep NaNs for imputers

print("X_reg:", X_reg.shape, "y_reg:", y_reg.shape)
print("Missing values:", int(X_reg.isna().sum().sum()))

display(X_reg.head(2))
display(y_reg.head(2))


X_reg: (2094, 40) y_reg: (2094,)
Missing values: 0


,No_1_Angle_Deviation,No_2_Angle_Deviation,No_3_Angle_Deviation,No_4_Angle_Deviation,No_5_Angle_Deviation,No_6_Angle_Deviation,No_7_Angle_Deviation,No_8_Angle_Deviation,No_9_Angle_Deviation,No_10_Angle_Deviation,...,No_18_NASM_Deviation,No_19_NASM_Deviation,No_20_NASM_Deviation,No_21_NASM_Deviation,No_22_NASM_Deviation,No_23_NASM_Deviation,No_24_NASM_Deviation,No_25_NASM_Deviation,No_1_Time_Deviation,No_2_Time_Deviation
0,0.538020,0.815878,0.346724,0.382114,0.302248,0.947872,0.275945,0.52176,0.457198,0.210426,...,0.587757,0.833094,0.656624,0.642276,0.552846,0.648972,0.578192,0.560019,0.821616,0.818747
1,0.443807,0.306552,0.823529,0.188905,0.497370,0.140124,0.664275,0.52176,0.729316,0.912482,...,0.441416,0.670971,0.721186,0.826399,0.805356,0.848876,0.889048,0.816834,0.307987,0.248207


0    0.323667
1    0.323699
Name: AimoScore, dtype: float64

## Define a baseline model to train on each bootstrap sample

In [23]:
from sklearn.linear_model import Ridge

# Simple, stable baseline model (fast for many bootstraps)
base_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", Ridge(alpha=1.0, random_state=RANDOM_SEED))
])

base_model


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a fe

## Train/test split (fixed) + bootstrap configuration

In [24]:
# Fixed split (bootstrap happens ONLY on the training split)
X_train, X_test, y_train, y_test = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_SEED
)

print("Train:", X_train.shape, y_train.shape)
print("Test :", X_test.shape, y_test.shape)

# Bootstrap config
N_BOOTSTRAPS = 25          # number of bootstrap datasets/models
BOOTSTRAP_SEED0 = 10_000   # base seed so it's reproducible

# Where we’ll save artifacts before logging them
ART_DIR = safe_mkdir(Path("./_mlflow_artifacts_bootstrap"))


Train: (1675, 40) (1675,)
Test : (419, 40) (419,)


## Bootstrap loop: create quasi-independent datasets + log each run to MLflow

In [25]:
from sklearn.base import clone

bootstrap_test_metrics: List[Dict[str, float]] = []

with mlflow.start_run(run_name="bootstrap_parent") as parent_run:
    mlflow.log_params({
        "random_seed": RANDOM_SEED,
        "n_bootstraps": N_BOOTSTRAPS,
        "bootstrap_seed0": BOOTSTRAP_SEED0,
        "train_size": int(X_train.shape[0]),
        "test_size": int(X_test.shape[0]),
        "model_class": type(base_model.named_steps["model"]).__name__,
    })

    for b in range(N_BOOTSTRAPS):
        seed = BOOTSTRAP_SEED0 + b
        n_train = X_train.shape[0]

        boot_idx = make_bootstrap_indices(n_train, seed=seed)
        Xb = X_train.iloc[boot_idx]
        yb = y_train.iloc[boot_idx]

        # Out-of-bag (OOB) evaluation on training split (optional but useful)
        mask_oob = oob_mask(n_train, boot_idx)
        X_oob = X_train.loc[mask_oob]
        y_oob = y_train.loc[mask_oob]

        m = clone(base_model)

        with mlflow.start_run(run_name=f"bootstrap_{b:03d}", nested=True):
            # Log bootstrap properties
            mlflow.log_params({
                "bootstrap_id": b,
                "bootstrap_seed": seed,
                "bootstrap_sample_size": int(len(boot_idx)),
                "bootstrap_unique_count": int(len(np.unique(boot_idx))),
                "oob_size": int(X_oob.shape[0]),
            })

            # Fit on bootstrap sample
            m.fit(Xb, yb)

            # Evaluate on test set (consistent across bootstraps)
            yhat_test = m.predict(X_test)
            test_m = regression_metrics(y_test, yhat_test)

            # Optional: evaluate on OOB (if any OOB points exist)
            if X_oob.shape[0] > 0:
                yhat_oob = m.predict(X_oob)
                oob_m = regression_metrics(y_oob, yhat_oob)
                for k, v in oob_m.items():
                    mlflow.log_metric(f"oob_{k}", v)

            for k, v in test_m.items():
                mlflow.log_metric(f"test_{k}", v)

            bootstrap_test_metrics.append(test_m)

            # Save bootstrap indices as artifact (so dataset is reproducible)
            idx_path = ART_DIR / f"bootstrap_indices_{b:03d}.npy"
            np.save(idx_path, boot_idx)
            mlflow.log_artifact(str(idx_path), artifact_path="bootstrap_indices")

            # Log model
            mlflow.sklearn.log_model(m, name="model")

    # Parent-level summary over bootstraps
    r2s = [m["r2"] for m in bootstrap_test_metrics]
    rmses = [m["rmse"] for m in bootstrap_test_metrics]
    maes = [m["mae"] for m in bootstrap_test_metrics]

    mlflow.log_metrics({
        "bootstrap_test_r2_mean": float(np.mean(r2s)),
        "bootstrap_test_r2_std": float(np.std(r2s)),
        "bootstrap_test_rmse_mean": float(np.mean(rmses)),
        "bootstrap_test_rmse_std": float(np.std(rmses)),
        "bootstrap_test_mae_mean": float(np.mean(maes)),
        "bootstrap_test_mae_std": float(np.std(maes)),
    })

    print("Parent run:", parent_run.info.run_id)
    print("Bootstrap test R2 mean/std:", float(np.mean(r2s)), float(np.std(r2s)))
    print("Bootstrap test RMSE mean/std:", float(np.mean(rmses)), float(np.std(rmses)))
    print("Bootstrap test MAE mean/std:", float(np.mean(maes)), float(np.std(maes)))


🏃 View run bootstrap_000 at: https://172.27.62.170:5001/#/experiments/4/runs/5e9c3e06c324449eb993940bc892df86
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/4
🏃 View run bootstrap_001 at: https://172.27.62.170:5001/#/experiments/4/runs/716805026af044feaf475041ac019768
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/4
🏃 View run bootstrap_002 at: https://172.27.62.170:5001/#/experiments/4/runs/286b4b1d06b94842aa79f38c0e9def65
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/4
🏃 View run bootstrap_003 at: https://172.27.62.170:5001/#/experiments/4/runs/17ecdf8715054a39ab0dd5bb330e0d6a
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/4
🏃 View run bootstrap_004 at: https://172.27.62.170:5001/#/experiments/4/runs/395831e28dac4649a7594b70a4f8756f
🧪 View experiment at: https://172.27.62.170:5001/#/experiments/4
🏃 View run bootstrap_005 at: https://172.27.62.170:5001/#/experiments/4/runs/74f36d6f1d9c480aa1ec1d1e654c62a3
🧪 View experime

## Local summary table

In [26]:
summary_df = pd.DataFrame(bootstrap_test_metrics)
display(summary_df.describe().T)


,count,mean,std,min,25%,50%,75%,max
r2,25.0,0.607144,0.007681,0.591063,0.603287,0.606364,0.610388,0.624068
rmse,25.0,0.153147,0.001500,0.149819,0.152521,0.153306,0.153905,0.156258
mae,25.0,0.116403,0.001415,0.113451,0.115542,0.116400,0.117383,0.119883
